### Importa los datos
Dataset con diferentes fármacos, sus efectos y ratings de los clientes.

Importa el dataset *drugLibTrain_raw.tsv*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
df = pd.read_csv("data/drugLibTrain_raw.tsv", sep="\t")
df

,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,3947,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,1951,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above
...,...,...,...,...,...,...,...,...,...
3102,1039,vyvanse,10,Highly Effective,Mild Side Effects,adhd,"Increased focus, attention, productivity. Bett...","Restless legs at night, insomnia, headache (so...","I took adderall once as a child, and it made m..."
3103,3281,zoloft,1,Ineffective,Extremely Severe Side Effects,depression,Emotions were somewhat blunted. Less moodiness.,"Weight gain, extreme tiredness during the day,...",I was on Zoloft for about 2 years total. I am ...
3104,1664,climara,2,Marginally Effective,Moderate Side Effects,total hysterctomy,---,Constant issues with the patch not staying on....,---
3105,2621,trileptal,8,Considerably Effective,Mild Side Effects,epilepsy,Controlled complex partial seizures.,"Dizziness, fatigue, nausea",Started at 2 doses of 300 mg a day and worked ...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3107 entries, 0 to 3106
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         3107 non-null   int64 
 1   urlDrugName        3107 non-null   object
 2   rating             3107 non-null   int64 
 3   effectiveness      3107 non-null   object
 4   sideEffects        3107 non-null   object
 5   condition          3106 non-null   object
 6   benefitsReview     3089 non-null   object
 7   sideEffectsReview  3032 non-null   object
 8   commentsReview     3095 non-null   object
dtypes: int64(2), object(7)
memory usage: 218.6+ KB


In [4]:
df.describe()

,Unnamed: 0,rating
count,3107.000000,3107.000000
mean,2080.607016,7.006115
std,1187.998828,2.937582
min,0.000000,1.000000
25%,1062.500000,5.000000
50%,2092.000000,8.000000
75%,3092.500000,9.000000
max,4161.000000,10.000000


In [5]:
df.columns

Index(['Unnamed: 0', 'urlDrugName', 'rating', 'effectiveness', 'sideEffects',
       'condition', 'benefitsReview', 'sideEffectsReview', 'commentsReview'],
      dtype='object')

### Descriptive Analysis

Quedate únicamente con las columnas que podamos manejar: Columnas numéricas y columnas categóricas con pocas categorías (menos de 10)

In [6]:
# Eliminar la columna 'Unnamed: 0', no aporta valor predictivo
df_= df.drop(columns=['Unnamed: 0'], inplace = True)
df

,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above
...,...,...,...,...,...,...,...,...
3102,vyvanse,10,Highly Effective,Mild Side Effects,adhd,"Increased focus, attention, productivity. Bett...","Restless legs at night, insomnia, headache (so...","I took adderall once as a child, and it made m..."
3103,zoloft,1,Ineffective,Extremely Severe Side Effects,depression,Emotions were somewhat blunted. Less moodiness.,"Weight gain, extreme tiredness during the day,...",I was on Zoloft for about 2 years total. I am ...
3104,climara,2,Marginally Effective,Moderate Side Effects,total hysterctomy,---,Constant issues with the patch not staying on....,---
3105,trileptal,8,Considerably Effective,Mild Side Effects,epilepsy,Controlled complex partial seizures.,"Dizziness, fatigue, nausea",Started at 2 doses of 300 mg a day and worked ...


In [25]:
# Columnas numéricas
df_numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
df_numeric_cols

['rating']

In [26]:

# Columnas categóricas (con menos de 10 categorías)
df_cat_cols_small = [col for col in df.select_dtypes(include='object').columns 
                  if df[col].nunique() < 10]
df_cat_cols_small

['effectiveness', 'sideEffects']

In [27]:
# Nuevo Dataframe
df_cols_finales = df_numeric_cols + df_cat_cols_small


print("Columnas seleccionadas:", df_cols_finales)


Columnas seleccionadas: ['rating', 'effectiveness', 'sideEffects']


#### Transforma las columnas categóricas

Transforma las columnas categoricas a numericas mediante dummies

In [28]:
# Pasar de variables categóricas a numéricas (usando One-Hot Encoding)
df_encoded = pd.get_dummies(df_cols_finales, columns=['effectiveness', 'sideEffects'])

In [29]:
# Escalar los datos (StandardScaler)
# K-Means es sensible a la escala, por lo que es fundamental normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

In [30]:
# Mostramos cómo queda el array o el dataframe transformado
print("Dimensiones del dataset transformado:", X_scaled.shape)
pd.DataFrame(X_scaled, columns = df_encoded.columns).head()

Dimensiones del dataset transformado: (3, 3)


,effectiveness,rating,sideEffects
0,-0.707107,1.414214,-0.707107
1,1.414214,-0.707107,-0.707107
2,-0.707107,-0.707107,1.414214


#### Evalua cual es la mejor K

Utiliza silhouette_score para evaluar cual es la mejor K.

In [31]:
silhouette_scores = []
k_range = range(2, 11)

# Calculamos el score para cada K
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

# Graficamos los resultados
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, marker='o', linestyle='--')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Evaluación de K mediante Silhouette Score')
plt.grid(True)
plt.show()

# Imprimimos la mejor K
best_k = k_range[np.argmax(silhouette_scores)]
print(f"La mejor K según el Silhouette Score es: {best_k}")

ValueError: Number of labels is 3. Valid values are 2 to n_samples - 1 (inclusive)

#### Genera el K Means 

Comprueba los resultados y muestra en un pie plot la distribución de los distintos clusters.

In [33]:

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(X_scaled)

NameError: name 'best_k' is not defined

In [36]:

df_cols_finales = df_cols_finales.copy() # Copia para evitar advertencias
df_cols_finales['Cluster'] = kmeans_final.labels_

NameError: name 'kmeans_final' is not defined

In [37]:

cluster_counts = df_cols_finales['Cluster'].value_counts().sort_index()

TypeError: list indices must be integers or slices, not str

In [39]:
plt.figure(figsize=(8, 8))
plt.pie(cluster_counts, 
        labels=[f'Cluster {i}' for i in cluster_counts.index], 
        autopct='%1.1f%%', 
        startangle=140,
        colors=sns.color_palette('pastel'))
plt.title(f'Distribución de los {best_k} Clusters')
plt.show()

NameError: name 'cluster_counts' is not defined

<Figure size 800x800 with 0 Axes>